# UD7.03. El coste del error y el umbral de derivación

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
Bloques 5 y 6 de los apuntes · Criterio **4.c**

---

El cuaderno anterior terminó con un modelo del 94,4 %. Y ahora la pregunta que nadie
contesta con una exactitud:

> **¿Automatizamos?**

No. La pregunta correcta es **«¿a partir de qué confianza?»**, y tiene respuesta
numérica. Este cuaderno la calcula.

Al final tienes que poder contestar:

1. ¿Por qué una exactitud media no decide nada?
2. ¿Cuánto cuesta automatizarlo todo frente a automatizar solo lo que el modelo sabe?
3. ¿Por qué el umbral que elige el coste medio **no** es el que elige el riesgo?
4. ¿Se puede leer la confianza del modelo como una probabilidad? (Cuidado con esta.)

In [ ]:
import os
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, hstack
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

pd.set_option("display.width", 130)
SEMILLA = 20262027

BASE = ("https://raw.githubusercontent.com/RafaSalaEsteve/IABD-PIA-notebooks"
        "/main/UD7/datos/")
os.makedirs("datos", exist_ok=True)
RUTA = os.path.join("datos", "buzon_techstore.csv")
if not os.path.exists(RUTA):
    urllib.request.urlretrieve(BASE + "buzon_techstore.csv", RUTA)

buzon = pd.read_csv(RUTA, parse_dates=["fecha_hora"])
print(f"{len(buzon):,} mensajes")

## 1. El coste del error no es un número: es una matriz

Encaminar mal un mensaje de publicidad cuesta unos minutos. Encaminar mal una garantía
cuesta un escalado, una sustitución y el riesgo de una reclamación de consumo. Sumar los
dos como si fueran «un error» es lo que hace que una automatización parezca buena cuando
no lo es.

Y hay una asimetría que pesa más que los precios: **si un mensaje acaba en la bandeja de
publicidad, no lo vuelve a mirar nadie.** Ese error no se arregla nunca, y por eso
cuesta 2,5 veces más que el mismo error en cualquier otra dirección.

In [ ]:
COSTE_ERROR = {"spam": 1.50, "envio": 8.00, "devolucion": 18.00,
               "factura": 25.00, "producto": 35.00, "garantia": 120.00}
PENALIZACION_A_SPAM = 2.5

CATEGORIAS = sorted(COSTE_ERROR)

# La matriz completa, que es lo que hay que entregar en la P7.1: filas la categoría
# real, columnas a dónde se encaminó, y en cada celda lo que cuesta ese error concreto.
matriz_coste = pd.DataFrame(
    [[0.0 if real == puesta
      else COSTE_ERROR[real] * (PENALIZACION_A_SPAM if puesta == "spam" else 1.0)
      for puesta in CATEGORIAS] for real in CATEGORIAS],
    index=CATEGORIAS, columns=CATEGORIAS)
matriz_coste.index.name = "era"
matriz_coste.columns.name = "se encaminó a"
print(matriz_coste)
print()
print(f"El error más caro del proceso: garantía que acaba en la papelera, "
      f"{matriz_coste.loc['garantia', 'spam']:.0f} EUR.")
print(f"El más barato: publicidad tratada como consulta, "
      f"{matriz_coste.loc['spam', 'producto']:.2f} EUR.")
print(f"Entre uno y otro hay un factor de "
      f"{matriz_coste.loc['garantia', 'spam'] / matriz_coste.loc['spam', 'producto']:.0f}.")

> **Doscientas veces.** Ése es el factor entre el error más caro y el más barato de este
> proceso. Cualquier decisión tomada sobre la exactitud media está ignorando un factor de
> doscientos.

## 2. El modelo, y su confianza

Montamos el mismo modelo del cuaderno anterior, pero ahora nos interesa lo que devuelve
`predict_proba`: no solo qué categoría, sino **cuánta seguridad tiene**.

Y una decisión metodológica que hay que entender, porque es la diferencia entre elegir
un umbral y sobreajustarlo:

> **El umbral NO se elige sobre los 627 mensajes de la partición temporal.** Con esa
> muestra la curva de coste sale a saltos y el mínimo se mueve de sitio según la semilla.
> Se usan los 2.400 del año con **predicción fuera de pliegue**: cada mensaje lo predice
> un modelo que no lo ha visto al entrenar.

La partición temporal del cuaderno anterior sirve para **estimar cómo se comportará el
modelo**; la validación cruzada sirve para **elegir un parámetro con suficientes datos**.
Son dos preguntas distintas y se contestan con particiones distintas.

In [ ]:
def contexto(datos):
    dias = datos["dias_desde_compra"] if hasattr(datos, "columns") else datos
    return np.c_[dias.fillna(-1) / 100.0,
                 (dias <= 14).fillna(False).astype(int),
                 dias.isna().astype(int)]


tuberia = Pipeline([
    ("juntar", ColumnTransformer([
        ("texto", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), "texto"),
        ("contexto", FunctionTransformer(contexto, validate=False), ["dias_desde_compra"]),
    ])),
    ("modelo", LogisticRegression(max_iter=3000, C=5)),
])

# El vectorizador va DENTRO de la tubería a propósito: si se ajustara fuera, vería el
# vocabulario del pliegue de prueba y la estimación saldría optimista.
probabilidades = cross_val_predict(
    tuberia, buzon, buzon["categoria"], method="predict_proba",
    cv=StratifiedKFold(5, shuffle=True, random_state=SEMILLA))

clases = np.array(sorted(buzon["categoria"].unique()))
prediccion = clases[probabilidades.argmax(axis=1)]
confianza = probabilidades.max(axis=1)
verdad = buzon["categoria"].to_numpy()
persona = buzon["categoria_asignada"].to_numpy()   # lo que puso la persona, para comparar

print(f"Exactitud fuera de pliegue: {float((prediccion == verdad).mean()):.1%}")
print(f"Confianza mediana:          {np.median(confianza):.3f}")
print(f"Mensajes con confianza < 0,70: {float((confianza < 0.70).mean()):.1%}")

### ¿Se puede leer esa confianza como una probabilidad?

Es la pregunta de la UD4, y aquí importa mucho: si vamos a poner un umbral en 0,70,
conviene saber si «0,70» significa «acierta el 70 % de las veces».

El instrumento es el mismo de la UD4: un **diagrama de fiabilidad**. Se agrupan las
predicciones por tramos de confianza y se compara la confianza media de cada tramo con
el acierto real de ese tramo.

In [ ]:
acierta = (prediccion == verdad).astype(int)
tramos = pd.cut(confianza, bins=[0, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0])
fiabilidad = pd.DataFrame({"confianza": confianza, "acierta": acierta, "tramo": tramos})
resumen = fiabilidad.groupby("tramo", observed=True).agg(
    n=("acierta", "size"), confianza_media=("confianza", "mean"),
    acierto_real=("acierta", "mean"))
print(resumen.round(3))

fig, eje = plt.subplots(figsize=(5.5, 5.5))
eje.plot([0, 1], [0, 1], "--", color="#9AA5B1", label="calibración perfecta")
eje.plot(resumen["confianza_media"], resumen["acierto_real"], "o-",
         color="#4878A8", label="el modelo")
for _, fila in resumen.iterrows():
    eje.annotate(f"n={int(fila['n'])}",
                 (fila["confianza_media"], fila["acierto_real"]),
                 textcoords="offset points", xytext=(6, -10), fontsize=8)
eje.set_xlabel("confianza que dice el modelo")
eje.set_ylabel("acierto real")
eje.set_title("Diagrama de fiabilidad")
eje.set_xlim(0.2, 1.02)
eje.set_ylim(0.2, 1.02)
eje.legend()
plt.tight_layout()
plt.show()

Mira la tabla antes de seguir. Si los puntos caen por encima de la diagonal, el modelo es
**pesimista**: acierta más de lo que dice. Si caen por debajo, es **optimista**: dice más
seguridad de la que tiene, y entonces un umbral de 0,70 no protege lo que creemos.

> **Esto no invalida el umbral**, porque el umbral se va a elegir midiendo el coste real
> a cada valor, no suponiendo que 0,70 significa 70 %. Pero hay que saberlo, porque la
> frase «el sistema solo automatiza cuando está seguro por encima del 70 %» significa
> cosas distintas según esta gráfica, y en un informe hay que decir cuál.

## 3. Las tres partidas de la cuenta

Para comparar dos umbrales hay que sumar tres cosas. Dejarse una es hacer trampa, y la
que se deja todo el mundo es la tercera:

1. **El personal**, que baja pero no desaparece: el triaje es el **35 %** del tiempo de
   tratar un mensaje; el otro 65 % —resolverlo y contestar— lo sigue haciendo alguien.
2. **Los errores de la máquina** en la parte que automatiza.
3. **Los errores de la persona** en la parte que se le deriva. Derivar no es gratis: la
   persona también falla, y además los casos que se le derivan son **los difíciles**.

In [ ]:
COSTE_HORA = 18.50
FRACCION_TRIAJE = 0.35     # supuesto declarado: qué parte del tiempo es triaje


def coste_de_los_errores(datos, prediccion_datos):
    real = datos["categoria"].to_numpy()
    fallos = prediccion_datos != real
    return sum(COSTE_ERROR[r] * (PENALIZACION_A_SPAM if p == "spam" else 1.0)
               for r, p in zip(real[fallos], prediccion_datos[fallos]))


def coste_anual(umbral):
    """Coste de un año de buzón con triaje automático por encima del umbral."""
    automatiza = confianza >= umbral
    minutos = buzon["minutos_manual"].to_numpy()

    horas = (minutos[automatiza].sum() * (1 - FRACCION_TRIAJE)
             + minutos[~automatiza].sum()) / 60
    personal = horas * COSTE_HORA
    errores = (coste_de_los_errores(buzon[automatiza], prediccion[automatiza])
               + coste_de_los_errores(buzon[~automatiza], persona[~automatiza]))
    return {
        "umbral": umbral,
        "cobertura": float(automatiza.mean()),
        "precision": float((prediccion[automatiza] == verdad[automatiza]).mean())
        if automatiza.any() else np.nan,
        "personal": personal,
        "errores": errores,
        "total": personal + errores,
    }


curva = pd.DataFrame([coste_anual(u) for u in
                      [0.0, 0.40, 0.50, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 0.99, 1.01]])
curva_mostrar = curva.copy()
curva_mostrar["umbral"] = curva_mostrar["umbral"].map(
    lambda u: "a mano" if u > 1 else f"{u:.2f}")
print(curva_mostrar.round(3).to_string(index=False))

In [ ]:
mejor = curva.loc[curva["total"].idxmin()]
todo_auto = curva.iloc[0]
a_mano = curva.iloc[-1]

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))

finitos = curva[curva["umbral"] <= 1]
ejes[0].plot(finitos["umbral"], finitos["personal"], "o-", label="personal", color="#4878A8")
ejes[0].plot(finitos["umbral"], finitos["errores"], "o-", label="errores", color="#B4453C")
ejes[0].plot(finitos["umbral"], finitos["total"], "o-", label="total", color="#2F2F2F", linewidth=2)
ejes[0].axvline(mejor["umbral"], color="#3B7A57", linestyle="--",
                label=f"óptimo {mejor['umbral']:.2f}")
ejes[0].axhline(a_mano["total"], color="#9AA5B1", linestyle=":",
                label=f"todo a mano {a_mano['total']:,.0f} EUR")
ejes[0].set_xlabel("umbral de confianza")
ejes[0].set_ylabel("euros al año")
ejes[0].set_title("Las tres partidas, y su suma")
ejes[0].legend(fontsize=8)

ejes[1].plot(finitos["cobertura"], finitos["precision"], "o-", color="#4878A8")
for _, f in finitos.iterrows():
    if f["cobertura"] > 0:
        ejes[1].annotate(f"{f['umbral']:.2f}", (f["cobertura"], f["precision"]),
                         textcoords="offset points", xytext=(5, -9), fontsize=8)
ejes[1].set_xlabel("cobertura: proporción que se automatiza")
ejes[1].set_ylabel("precisión de lo automatizado")
ejes[1].set_title("Cuanto más se automatiza, peor se automatiza")
plt.tight_layout()
plt.show()

print(f"Óptimo:                 umbral {mejor['umbral']:.2f}, cobertura "
      f"{mejor['cobertura']:.0%}, {mejor['total']:,.0f} EUR/año")
print(f"Automatizarlo todo:     {todo_auto['total']:,.0f} EUR/año "
      f"({todo_auto['total'] - mejor['total']:+,.0f} EUR)")
print(f"Dejarlo todo a mano:    {a_mano['total']:,.0f} EUR/año "
      f"({a_mano['total'] - mejor['total']:+,.0f} EUR)")

La curva tiene forma de U. En lugar de dar por buena la explicación de siempre, vamos a
medir **por qué** sube la ladera de la derecha, porque la razón es concreta y se puede
comprobar en una línea: derivar un mensaje solo sirve si **la persona lo hace mejor que
la máquina**, y eso deja de ser cierto a partir de cierto umbral.

In [ ]:
print(f"{'umbral':>7} {'derivados':>11} {'acierta la máquina':>20} {'acierta la persona':>20}")
for u in [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]:
    derivados = confianza < u
    if derivados.sum() == 0:
        continue
    print(f"{u:>7.2f} {derivados.sum():>11} "
          f"{float((prediccion[derivados] == verdad[derivados]).mean()):>19.1%} "
          f"{float((persona[derivados] == verdad[derivados]).mean()):>19.1%}")
print()
print("Ahí está la ladera de la derecha. Mientras la persona acierta más que la máquina")
print("en lo que se le deriva, derivar mejora el proceso. En cuanto el umbral sube lo")
print("suficiente, se le empiezan a derivar mensajes que la máquina acertaba, y la")
print("persona acierta MENOS que ella. A partir de ahí, derivar es pagar por empeorar.")

> **El humano en el bucle solo ayuda donde el humano es mejor.** Suena obvio escrito así,
> y sin embargo casi ningún diseño lo comprueba: se supone que una revisión humana
> siempre mejora, y no es verdad. Aquí el cruce está alrededor del 0,80, y es lo que
> explica que la curva de coste suba a partir de ahí.

> **Automatizarlo todo sin umbral cuesta más que automatizar el 91 %.**
>
> Es el resultado más contraintuitivo de la unidad y conviene tenerlo a mano: *más
> automatización no es mejor automatización*. Hay un punto a partir del cual cada caso
> que se automatiza de más cuesta dinero.

## 4. Y ahora el error caro, que ninguna media enseña

La tabla anterior es una media. Una media puede estar bien mientras el proceso es
inaceptable. La pregunta que de verdad se hace en una empresa es otra:

> **¿Qué pasa con las garantías?**

In [ ]:
es_garantia = verdad == "garantia"

print(f"{'umbral':>8} {'garantías mal':>15} {'a la papelera':>15} {'coste total':>13}")
for u in [0.0, 0.60, 0.70, 0.80, 0.90, 0.95]:
    automatiza = confianza >= u
    # Lo que hace la máquina donde automatiza, y la persona donde se deriva.
    final = np.where(automatiza, prediccion, persona)
    mal = int(((final != verdad) & es_garantia).sum())
    papelera = int((es_garantia & (final == "spam")).sum())
    fila = coste_anual(u)
    print(f"{u:>8.2f} {mal:>15} {papelera:>15} {fila['total']:>12,.0f} EUR")

mal_persona = int(((persona != verdad) & es_garantia).sum())
print(f"{'a mano':>8} {mal_persona:>15} "
      f"{int((es_garantia & (persona == 'spam')).sum()):>15} "
      f"{curva.iloc[-1]['total']:>12,.0f} EUR")
print()
print(f"Garantías al año en el buzón: {int(es_garantia.sum())}")

Aquí conviene pararse, porque **el resultado no es el que esperaba quien escribió este
cuaderno** y es más interesante que el que esperaba.

1. **El proceso manual pierde muchísimas más garantías que cualquier automatización.**
   Ése es el punto de referencia real, y es peor de lo que nadie habría supuesto sin
   medirlo.
2. **Automatizar a ciegas manda garantías a la papelera**, de donde no las saca nadie.
   Aunque la media salga bien, eso basta por sí solo para descartar el umbral 0: son
   errores **irrecuperables**, y un error irrecuperable no se compensa con una media.
3. **Y lo que no esperábamos: subir el umbral por encima del óptimo NO salva más
   garantías.** Se quedan igual o empeoran. La razón es la que acabamos de medir: lo que
   se deriva de más va a una persona que en esos casos **no** acierta más que la máquina.

> **La conclusión correcta, entonces, no es «si el error es caro, sube el umbral».** Es
> **«si el error es caro, comprueba quién lo hace mejor en esos casos concretos»**. Aquí
> el umbral sirve para una cosa muy precisa —eliminar los errores irrecuperables de la
> papelera— y no sirve para lo que todo el mundo supone.
>
> Ésta es la diferencia entre evaluar y repetir lo que se espera oír. La primera versión
> de este cuaderno afirmaba que el umbral del riesgo era más alto que el del coste;
> medirlo con el proceso completo —máquina **y** persona— demostró que no. Se ha dejado
> escrito el error a propósito: es exactamente lo que el criterio 4.c pide evitar.

In [ ]:
# Lo que sí arregla el umbral, medido aparte: los errores IRRECUPERABLES.
print("Mensajes que no eran publicidad y acaban en la papelera, al año:")
for u in [0.0, 0.40, 0.50, 0.60, 0.70]:
    automatiza = confianza >= u
    final = np.where(automatiza, prediccion, persona)
    papelera = int(((verdad != "spam") & (final == "spam")).sum())
    caras = int(((verdad == "garantia") & (final == "spam")).sum())
    print(f"  umbral {u:.2f}: {papelera:>3} en total, {caras} de ellas garantías")
print()
print("Con un umbral tan bajo como 0,40 ya desaparecen los irrecuperables, y eso")
print("cuesta muy poco. Es la parte del umbral que sí hace lo que se supone.")

> **Así se informa una decisión de este tipo, y es lo que PR7 evalúa:**
>
> «Recomiendo el umbral 0,70. Es el mínimo de coste (8.645 euros al año frente a 18.805
> del proceso manual) y **además** elimina los errores irrecuperables, que es la
> condición que la dirección puso. Subirlo por encima no salva ni una garantía más: lo
> he medido, y lo que se deriva de más va a una persona que en esos casos no acierta más
> que el sistema. Si la dirección prefiere subirlo de todas formas por una razón que no
> sea la exactitud —por ejemplo, para que haya intervención humana declarada en más
> casos—, cuesta 285 euros al año pasar a 0,80 y hay que decirlo así.»
>
> Eso es **evaluar la conveniencia**. Decir «pondremos un umbral alto para ir sobre
> seguro» no lo es: no tiene número, no tiene precio y no se puede discutir.

## 5. Lo que se deriva no es una muestra al azar

Un último detalle que casi siempre se pasa por alto al dimensionar el equipo que queda.

In [ ]:
umbral = mejor["umbral"]
derivados = buzon[confianza < umbral]
automatizados = buzon[confianza >= umbral]

print(f"Con el umbral {umbral:.2f} se derivan {len(derivados)} mensajes al año "
      f"({len(derivados) / len(buzon):.1%})")
print()
print(f"{'':<26} {'derivados':>12} {'automatizados':>15}")
print(f"{'minutos por mensaje':<26} {derivados['minutos_manual'].mean():>12.1f} "
      f"{automatizados['minutos_manual'].mean():>15.1f}")
print(f"{'% confusos':<26} {derivados['confuso'].mean():>11.1%} "
      f"{automatizados['confuso'].mean():>14.1%}")
print(f"{'% de frontera':<26} {derivados['frontera'].mean():>11.1%} "
      f"{automatizados['frontera'].mean():>14.1%}")
print(f"{'acierto de la persona':<26} "
      f"{(derivados['categoria'] == derivados['categoria_asignada']).mean():>11.1%} "
      f"{(automatizados['categoria'] == automatizados['categoria_asignada']).mean():>14.1%}")
print()
print("Los mensajes que se derivan son más largos, más confusos y la persona acierta")
print("menos en ellos. La carga que queda NO es el 9 % del trabajo: es más por caso, y")
print("además es la parte desagradable. Dimensionar el equipo que queda con una regla")
print("de tres es el error clásico de un plan de automatización.")

> **Y hay un efecto que no sale en ninguna hoja de cálculo:** una persona que revisa
> decisiones correctas el 97 % de las veces **deja de revisarlas de verdad**. Si el humano
> en el bucle se convierte en alguien que pulsa «aceptar», la garantía que compramos con
> el umbral no existe, y además el proceso vuelve a caer dentro del artículo 22 del RGPD,
> que exige que la intervención humana sea real.
>
> Que la supervisión sea real —muestreo, rotación, un indicador de cuántas veces se
> cambia la propuesta del sistema— es parte del diseño, no un detalle de organización.

## 6. Lo que hay que llevarse

In [ ]:
print(f"""
LA DECISIÓN, EN CINCO LÍNEAS
============================
Todo a mano                 {curva.iloc[-1]['total']:>9,.0f} EUR/año
Todo automático sin umbral  {todo_auto['total']:>9,.0f} EUR/año
Automático con umbral {mejor['umbral']:.2f}  {mejor['total']:>9,.0f} EUR/año   <- el óptimo de coste
  cobertura                 {mejor['cobertura']:>9.0%}
  precisión de lo automatizado {mejor['precision']:>6.1%}
""")

1. **Un error no cuesta «un error».** Entre el más caro y el más barato de este proceso
   hay un factor de doscientos.
2. **Una exactitud no es una decisión.** La decisión es un umbral, y sale de sumar tres
   partidas, no de mirar un porcentaje.
3. **Más automatización no es mejor automatización.** La curva de coste tiene forma de U.
4. **La media no protege del error irrecuperable**, y el umbral sirve sobre todo para
   eliminarlo. Lo que **no** hace es salvar más errores caros por el hecho de subirlo:
   eso hay que medirlo, no suponerlo.
5. **Lo que se deriva no es una muestra al azar:** es lo difícil, y cuesta más por caso.

Falta una cosa para poder decidir: **cuánto cuesta construir todo esto**. Es el cuaderno
`UD7_04`.